In [ ]:
from typing import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph import graph
from langgraph.graph import StateGraph,START,END
from langgraph.errors import GraphRecursionError
from loguru import  logger
from rich import print

#1. 状態を宣言
class EmptyState(TypedDict):
    pass

#2. ノードを宣言
def loop_node(state:EmptyState,config:RunnableConfig) -> EmptyState:
    # 現在のスーパーステップ数を取得
    cur_step = config["metadata"]["langgraph_step"]
    logger.info("loop_node,cur_step:{}",cur_step)
    

#3. グラフを構築
builder = StateGraph(state_schema=EmptyState)

builder.add_node("loop_node",loop_node)

builder.add_edge(START,"loop_node")
builder.add_edge("loop_node","loop_node")
graph = builder.compile()

try:
    graph.invoke({},config={"recursion_limit":10})
except GraphRecursionError as e:
    logger.info("スーパーステップ数が上限に達しました。例外をスロー:{}",e)



In [ ]:
from IPython.display import display
display(graph)